In [1]:
import torch
import torchcrop
from torchcrop.utils.io import make_constant_weather

weather = make_constant_weather(batch_size=2, n_days=150)
model = torchcrop.Lintul5Model()
output = model(weather, start_doy=60)

print(output.yield_)  # [B] final storage-organ biomass (g m-2)
print(output.lai.shape)  # [B, T+1] LAI trajectory
print(output.dvs.shape)  # [B, T+1] development stage trajectory

tensor([916.7202, 916.7202])
torch.Size([2, 151])
torch.Size([2, 151])


In [ ]:
import torch
import torch.nn as nn
from torchcrop import Lintul5Model, CropParameters

crop = CropParameters().to(dtype=torch.float64)
crop.tsum1 = nn.Parameter(torch.tensor(900, dtype=torch.float64))

model = Lintul5Model(crop_params=crop).double()
optimizer = torch.optim.Adam([crop.rue], lr=1e-1)

observed_yield = 1200

# Add gradient inspection
for i in range(50):
    optimizer.zero_grad()
    out = model(weather.to(torch.float64), start_doy=60)
    loss = ((out.yield_ - observed_yield) ** 2).mean()
    loss.backward()

    # Check gradient magnitude
    grad = crop.tsum1.grad
    print(f"Iter {i}: loss={loss:.6f}, yield={out.yield_.mean():.2f}, grad={grad:.6e}")

    optimizer.step()

Iter 0: loss=80193.629776, yield=916.82, grad=6.030610e+02
Iter 1: loss=80193.629776, yield=916.82, grad=1.206122e+03
Iter 2: loss=80193.629776, yield=916.82, grad=1.809183e+03
Iter 3: loss=80193.629776, yield=916.82, grad=2.412244e+03
Iter 4: loss=80193.629776, yield=916.82, grad=3.015305e+03
Iter 5: loss=80193.629776, yield=916.82, grad=3.618366e+03
Iter 6: loss=80193.629776, yield=916.82, grad=4.221427e+03
Iter 7: loss=80193.629776, yield=916.82, grad=4.824488e+03
Iter 8: loss=80193.629776, yield=916.82, grad=5.427549e+03
Iter 9: loss=80193.629776, yield=916.82, grad=6.030610e+03
Iter 10: loss=80193.629776, yield=916.82, grad=6.633670e+03
Iter 11: loss=80193.629776, yield=916.82, grad=7.236731e+03
Iter 12: loss=80193.629776, yield=916.82, grad=7.839792e+03
Iter 13: loss=80193.629776, yield=916.82, grad=8.442853e+03


KeyboardInterrupt: 